In [3]:
!pip install geopy

  Using cached geopy-2.4.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached geographiclib-2.1-py3-none-any.whl.metadata (1.6 kB)
Using cached geopy-2.4.1-py3-none-any.whl (125 kB)
Using cached geographiclib-2.1-py3-none-any.whl (40 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [geopy]


In [4]:
import pandas as pd
import numpy as np
import sqlite3
import os
import time
from geopy.geocoders import Nominatim
from geopy.distance import geodesic

In [5]:
df = pd.read_csv('/Users/seoyoonham/Projects/fraud_detection/data/CustomerBehavior.csv')
conn = sqlite3.connect(':memory:')
df.to_sql('customer_behavior', conn, index=False, if_exists='replace')

50000

In [6]:
# Default query
query_default = """
SELECT *
FROM customer_behavior
WHERE User_ID IN (
    SELECT User_ID 
    FROM customer_behavior 
    GROUP BY User_ID 
    HAVING COUNT(Transaction_ID) >= 2
)
ORDER BY User_ID, Timestamp;
"""

df_default = pd.read_sql(query_default, conn).copy()

In [7]:
# Add helper column 'risk_score_new' 
df_default['risk_score_new'] = 0.0

# Data preprocess
df_default['Timestamp'] = pd.to_datetime(df_default['Timestamp'])

df_default.head()

,Transaction_ID,User_ID,Transaction_Amount,Transaction_Type,Timestamp,Account_Balance,Device_Type,Location,Merchant_Category,IP_Address_Flag,...,Avg_Transaction_Amount_7d,Failed_Transaction_Count_7d,Card_Type,Card_Age,Transaction_Distance,Authentication_Method,Risk_Score,Is_Weekend,Fraud_Label,risk_score_new
0,TXN_36059,USER_1000,69.02,Online,2023-02-21 10:25:00,36077.57,Tablet,London,Groceries,0,...,225.91,4,Amex,170,2983.85,Password,0.5529,0,1,0.0
1,TXN_28650,USER_1000,191.87,Online,2023-04-26 23:17:00,4201.83,Laptop,Tokyo,Electronics,0,...,357.91,2,Discover,14,3616.72,OTP,0.1897,0,0,0.0
2,TXN_40117,USER_1000,64.78,Online,2023-09-01 20:09:00,25291.69,Tablet,London,Clothing,0,...,106.00,2,Discover,133,2042.18,Password,0.9632,1,1,0.0
3,TXN_13680,USER_1000,383.60,Bank Transfer,2023-10-29 03:47:00,53558.88,Laptop,Mumbai,Clothing,0,...,441.82,1,Amex,3,567.59,OTP,0.1279,0,0,0.0
4,TXN_21630,USER_1000,39.41,Bank Transfer,2023-12-03 06:43:00,70995.83,Laptop,Mumbai,Clothing,0,...,120.49,1,Visa,73,4851.37,PIN,0.6365,1,0,0.0


In [8]:
# Column reordering function for better readability
def reorder_columns(df):
    cols = df.columns.tolist()

    # 1. Location-related keywords
    loc_related = ['lat', 'lon', 'prev_location', 'prev_lat', 'prev_lon', 'distance']
    
    # Extract existing columns
    existing_loc_cols = [col for col in loc_related if col in cols]
    
    for col in existing_loc_cols:
        cols.remove(col)
        
    if 'Location' in cols:
        loc_idx = cols.index('Location')
        for i, col in enumerate(existing_loc_cols):
            cols.insert(loc_idx + 1 + i, col)

    # 2. Time-related keywords
    time_related = ['prev_timestamp', 'time_diff_hours']

    # Extract existing columns
    existing_time_cols = [col for col in time_related if col in cols]
    
    for col in existing_time_cols:
        cols.remove(col)
        
    if 'Timestamp' in cols:
        ts_idx = cols.index('Timestamp')
        for i, col in enumerate(existing_time_cols):
            cols.insert(ts_idx + 1 + i, col)

    # 3. Count-related keywords
    count_related = ['avg_daily_count', 'std_daily_count']

    # Extract existing columns
    existing_count_cols = [col for col in count_related if col in cols]
    
    for col in existing_count_cols:
        cols.remove(col)
        
    if 'Daily_Transaction_Count' in cols:
        count_idx = cols.index('Daily_Transaction_Count')
        for i, col in enumerate(existing_count_cols):
            cols.insert(count_idx + 1 + i, col)
            
    # 4. Other keywords
    mapping = {
        'usual_device': 'Device_Type',
        'usual_auth': 'Authentication_Method',
        'avg_distance': 'Transaction_Distance'
    }
    
    for col, target in mapping.items():
        if col in cols and target in cols:
            cols.remove(col)
            cols.insert(cols.index(target) + 1, col)
    
    return df[cols]

In [9]:
# 1st Fraud feature : Impossible Travel
# Copy default dataframe
df_impos_travel = df_default.copy()

In [10]:
# Add time helper columns
df_impos_travel['prev_timestamp'] = df_impos_travel.groupby('User_ID')['Timestamp'].shift(1)
df_impos_travel['time_diff_hours'] = (df_impos_travel['Timestamp'] - df_impos_travel['prev_timestamp']).dt.total_seconds() / 3600

In [11]:
def get_city_coords(city_list, cache_file='city_coords.csv'):
    # Check if city_coords file exists
    if os.path.exists(cache_file):
        df_cache = pd.read_csv(cache_file)
        city_map = dict(zip(df_cache['Location'], zip(df_cache['lat'], df_cache['lon'])))
    else:
        city_map = {}

    # Filter new cities
    new_cities = [city for city in city_list if city not in city_map]
    
    if new_cities:
        print(f"Geocoding {len(new_cities)} new cities...")
        # Call API(Nominatim) to get coordination for new cities
        geolocator = Nominatim(user_agent="fraud_detector_v1")
        for city in new_cities:
            location = geolocator.geocode(city)
            if location:
                city_map[city] = (location.latitude, location.longitude)
            else:
                city_map[city] = (np.nan, np.nan)
            time.sleep(1) 

        # Update cache file
        df_updated_cache = pd.DataFrame([{'Location': k, 'lat': v[0], 'lon': v[1]} for k, v in city_map.items()])
        df_updated_cache.to_csv(cache_file, index=False)

    return pd.DataFrame([{'Location': k, 'lat': v[0], 'lon': v[1]} for k, v in city_map.items()])

In [12]:
def calculate_distance(row):
    # Handle missing coordinates 
    if pd.isna(row['prev_lat']) or pd.isna(row['lat']):
        return 0.0
    
    prev_coords = (row['prev_lat'], row['prev_lon'])
    current_coords = (row['lat'], row['lon'])

    # geodesic: Calculate actual shortest flight distance between 2 coordinates
    return geodesic(prev_coords, current_coords).miles

In [13]:
unique_cities = df_impos_travel['Location'].unique()
df_coords = get_city_coords(unique_cities)
df_impos_travel = pd.merge(df_impos_travel, df_coords, on='Location', how='left')

In [14]:
df_impos_travel['prev_location'] = df_impos_travel.groupby('User_ID')['Location'].shift(1)
df_impos_travel['prev_lat'] = df_impos_travel.groupby('User_ID')['lat'].shift(1)
df_impos_travel['prev_lon'] = df_impos_travel.groupby('User_ID')['lon'].shift(1)

In [15]:
df_impos_travel['distance'] = df_impos_travel.apply(calculate_distance, axis=1)

In [16]:
# Reorder columns
df_impos_travel = reorder_columns(df_impos_travel)

In [17]:
def check_impossible_travel(row):

    risk_increment = 0.0
    
    # No previous transaction 
    if pd.isna(row['prev_timestamp']) or row['time_diff_hours'] <= 0:
        return risk_increment
        
    if row['Location'] == row['prev_location']:
        return risk_increment

    # Calculate velocity
    v = row['distance'] / row['time_diff_hours']
    
    # Commercial jets generally fly between 500 to 600 miles per hour
    v_limit = 550 
    
    # Set sensitivity between 0.01 and 0.02
    k = 0.015 
    
    # Apply proportinal scoring with Sigmoid function 
    risk_score = 1 / (1 + np.exp(-k * (v - v_limit)))

    risk_increment = round(risk_score, 4)
    
    return risk_increment

In [18]:
df_impos_travel['risk_score_new'] += df_impos_travel.apply(check_impossible_travel, axis=1)

In [19]:
# Check if risk_score_new is updated properly
filtered_df = df_impos_travel[(df_impos_travel['risk_score_new'] > 0.5) & (df_impos_travel['Fraud_Label'] == 0)]
filtered_df

,Transaction_ID,User_ID,Transaction_Amount,Transaction_Type,Timestamp,prev_timestamp,time_diff_hours,Account_Balance,Device_Type,Location,...,Avg_Transaction_Amount_7d,Failed_Transaction_Count_7d,Card_Type,Card_Age,Transaction_Distance,Authentication_Method,Risk_Score,Is_Weekend,Fraud_Label,risk_score_new
287,TXN_45082,USER_1052,94.37,ATM Withdrawal,2023-01-01 21:59:00,2023-01-01 05:20:00,16.650000,24537.94,Tablet,London,...,128.21,2,Discover,94,3202.41,PIN,0.7606,0,0,0.7791
357,TXN_23111,USER_1063,7.31,Bank Transfer,2023-05-21 14:58:00,2023-05-21 01:45:00,13.216667,47213.52,Laptop,London,...,268.83,3,Mastercard,114,294.92,Password,0.6188,0,0,0.9766
480,TXN_48170,USER_1082,36.68,Bank Transfer,2023-02-22 17:14:00,2023-02-22 09:08:00,8.100000,12138.55,Tablet,Sydney,...,115.83,2,Visa,22,4889.40,Biometric,0.0005,0,0,1.0000
503,TXN_36754,USER_1087,11.52,Online,2023-06-02 19:13:00,2023-06-02 12:59:00,6.233333,14740.97,Laptop,New York,...,227.75,1,Discover,105,3081.84,Password,0.1646,0,0,0.9997
611,TXN_6475,USER_1105,13.06,POS,2023-03-11 02:47:00,2023-03-10 12:34:00,14.216667,23342.07,Mobile,Sydney,...,476.37,2,Amex,136,1081.06,OTP,0.7554,0,0,0.9031
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49151,TXN_6952,USER_9880,110.68,ATM Withdrawal,2023-12-07 05:39:00,2023-12-07 03:43:00,1.933333,34812.66,Tablet,London,...,430.35,2,Discover,163,805.29,OTP,0.7606,0,0,1.0000
49566,TXN_16005,USER_9962,48.77,Bank Transfer,2023-03-05 07:41:00,2023-03-05 03:12:00,4.483333,44185.87,Mobile,Tokyo,...,99.21,1,Visa,183,2202.22,OTP,0.0352,1,0,1.0000
49616,TXN_26341,USER_9970,31.87,Bank Transfer,2023-08-03 19:58:00,2023-08-03 18:43:00,1.250000,88204.53,Mobile,Sydney,...,57.70,2,Discover,144,674.57,OTP,0.2429,0,0,1.0000
49746,TXN_23080,USER_9992,336.21,ATM Withdrawal,2023-07-31 20:20:00,2023-07-31 17:08:00,3.200000,42599.37,Mobile,London,...,196.54,1,Discover,91,1649.61,Password,0.3174,0,0,1.0000


In [20]:
# 2nd Fraud Feature: High velocity
# Copy default dataframe
df_high_velocity = df_default.copy()

# Bring 'risk_score_new' from df_impos_travel and merge
df_high_velocity = pd.merge(
    df_high_velocity.drop(columns=['risk_score_new']), 
    df_impos_travel[['Transaction_ID', 'risk_score_new']],
    on='Transaction_ID',
    how='left'
)

In [21]:
df_high_velocity.head()

,Transaction_ID,User_ID,Transaction_Amount,Transaction_Type,Timestamp,Account_Balance,Device_Type,Location,Merchant_Category,IP_Address_Flag,...,Avg_Transaction_Amount_7d,Failed_Transaction_Count_7d,Card_Type,Card_Age,Transaction_Distance,Authentication_Method,Risk_Score,Is_Weekend,Fraud_Label,risk_score_new
0,TXN_36059,USER_1000,69.02,Online,2023-02-21 10:25:00,36077.57,Tablet,London,Groceries,0,...,225.91,4,Amex,170,2983.85,Password,0.5529,0,1,0.0000
1,TXN_28650,USER_1000,191.87,Online,2023-04-26 23:17:00,4201.83,Laptop,Tokyo,Electronics,0,...,357.91,2,Discover,14,3616.72,OTP,0.1897,0,0,0.0003
2,TXN_40117,USER_1000,64.78,Online,2023-09-01 20:09:00,25291.69,Tablet,London,Clothing,0,...,106.00,2,Discover,133,2042.18,Password,0.9632,1,1,0.0003
3,TXN_13680,USER_1000,383.60,Bank Transfer,2023-10-29 03:47:00,53558.88,Laptop,Mumbai,Clothing,0,...,441.82,1,Amex,3,567.59,OTP,0.1279,0,0,0.0003
4,TXN_21630,USER_1000,39.41,Bank Transfer,2023-12-03 06:43:00,70995.83,Laptop,Mumbai,Clothing,0,...,120.49,1,Visa,73,4851.37,PIN,0.6365,1,0,0.0000


In [22]:
# Calculate mean and standard deviation of daily transaction counts per user
user_stats = df_high_velocity.groupby('User_ID')['Daily_Transaction_Count'].agg(['mean', 'std']).reset_index()
user_stats.columns = ['User_ID', 'avg_daily_count', 'std_daily_count']

# Handle ZeroDivisionError and false positives cases
user_stats['std_daily_count'] = user_stats['std_daily_count'].fillna(0.5).replace(0, 0.5)

# Merge user_stats into the main dataframe
df_high_velocity = pd.merge(df_high_velocity, user_stats, on='User_ID', how='left')

In [23]:
df_high_velocity = reorder_columns(df_high_velocity)

In [24]:
def check_high_velocity(row):

    risk_increment = 0.0
    
    # No risk if today's transaction count is less than or equal to the user's average
    if row['Daily_Transaction_Count'] <= row['avg_daily_count']:
        return risk_increment
        
    # Calculate Z-Score 
    z_score = (row['Daily_Transaction_Count'] - row['avg_daily_count']) / row['std_daily_count']
    
    # Map Z-Score to a 0-1 risk score using exponential decay
    risk_score = 1 - np.exp(-0.5 * z_score)
    
    # Apply category-specific risk adjustments
    if row['Merchant_Category'] == 'Electronics':
        risk_score = min(risk_score + 0.15, 1.0) # Boost risk for high-target category, capped at 1.0
    elif row['Merchant_Category'] in ['Travel', 'Restaurants']:
        risk_score = max(risk_score - 0.05, 0.0) # Offset risk for low-suspicion everyday categories, floored at 0.0

    risk_increment = round(risk_score, 4)
    
    return risk_increment

# Update risk_score_new based on 2nd fraud detection feature
df_high_velocity['risk_score_new'] += df_high_velocity.apply(check_high_velocity, axis=1)

In [25]:
df_high_velocity[df_high_velocity['risk_score_new'] > 0]

,Transaction_ID,User_ID,Transaction_Amount,Transaction_Type,Timestamp,Account_Balance,Device_Type,Location,Merchant_Category,IP_Address_Flag,...,Avg_Transaction_Amount_7d,Failed_Transaction_Count_7d,Card_Type,Card_Age,Transaction_Distance,Authentication_Method,Risk_Score,Is_Weekend,Fraud_Label,risk_score_new
1,TXN_28650,USER_1000,191.87,Online,2023-04-26 23:17:00,4201.83,Laptop,Tokyo,Electronics,0,...,357.91,2,Discover,14,3616.72,OTP,0.1897,0,0,0.2155
2,TXN_40117,USER_1000,64.78,Online,2023-09-01 20:09:00,25291.69,Tablet,London,Clothing,0,...,106.00,2,Discover,133,2042.18,Password,0.9632,1,1,0.4365
3,TXN_13680,USER_1000,383.60,Bank Transfer,2023-10-29 03:47:00,53558.88,Laptop,Mumbai,Clothing,0,...,441.82,1,Amex,3,567.59,OTP,0.1279,0,0,0.2105
6,TXN_31869,USER_1001,150.26,Online,2023-02-20 18:12:00,97356.04,Laptop,Tokyo,Clothing,0,...,131.40,0,Discover,54,2626.52,OTP,0.2736,0,0,0.0003
7,TXN_16009,USER_1001,9.65,Bank Transfer,2023-03-18 06:17:00,58740.69,Tablet,London,Electronics,0,...,199.32,3,Visa,28,501.77,PIN,0.6659,0,0,0.6679
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49799,TXN_36549,USER_9998,336.28,POS,2023-08-22 22:43:00,55490.35,Tablet,London,Electronics,0,...,346.64,0,Visa,41,3483.61,PIN,0.1581,1,0,1.0013
49801,TXN_20102,USER_9998,14.89,Online,2023-09-30 17:47:00,46190.69,Tablet,New York,Electronics,0,...,11.85,1,Mastercard,212,4546.61,OTP,0.7063,0,0,0.4778
49802,TXN_11163,USER_9998,49.53,Bank Transfer,2023-11-04 04:01:00,81626.81,Tablet,New York,Electronics,0,...,220.06,1,Amex,34,4727.77,OTP,0.2499,0,0,0.4775
49803,TXN_36472,USER_9998,228.91,Online,2023-12-13 09:28:00,34061.29,Mobile,Mumbai,Restaurants,0,...,181.33,3,Amex,222,3623.02,OTP,0.4687,0,0,0.0003


In [26]:
# 3rd Fraud feature : Small to Large Transation
# Copy default dataframe
df_small_large = df_default.copy()

df_small_large['prev_timestamp'] = df_small_large.groupby('User_ID')['Timestamp'].shift(1)
df_small_large['time_diff_hours'] = (df_small_large['Timestamp'] - df_small_large['prev_timestamp']).dt.total_seconds() / 3600
df_small_large['prev_amount'] = df_small_large.groupby('User_ID')['Transaction_Amount'].shift(1)

df_small_large = reorder_columns(df_small_large)

# Bring 'risk_score_new' from df_high_velocity and merge
df_small_large = pd.merge(
    df_small_large.drop(columns=['risk_score_new']), 
    df_high_velocity[['Transaction_ID', 'risk_score_new']],
    on='Transaction_ID',
    how='left'
)

df_small_large.head()

,Transaction_ID,User_ID,Transaction_Amount,Transaction_Type,Timestamp,prev_timestamp,time_diff_hours,Account_Balance,Device_Type,Location,...,Failed_Transaction_Count_7d,Card_Type,Card_Age,Transaction_Distance,Authentication_Method,Risk_Score,Is_Weekend,Fraud_Label,prev_amount,risk_score_new
0,TXN_36059,USER_1000,69.02,Online,2023-02-21 10:25:00,NaT,NaN,36077.57,Tablet,London,...,4,Amex,170,2983.85,Password,0.5529,0,1,NaN,0.0000
1,TXN_28650,USER_1000,191.87,Online,2023-04-26 23:17:00,2023-02-21 10:25:00,1548.866667,4201.83,Laptop,Tokyo,...,2,Discover,14,3616.72,OTP,0.1897,0,0,69.02,0.2155
2,TXN_40117,USER_1000,64.78,Online,2023-09-01 20:09:00,2023-04-26 23:17:00,3068.866667,25291.69,Tablet,London,...,2,Discover,133,2042.18,Password,0.9632,1,1,191.87,0.4365
3,TXN_13680,USER_1000,383.60,Bank Transfer,2023-10-29 03:47:00,2023-09-01 20:09:00,1375.633333,53558.88,Laptop,Mumbai,...,1,Amex,3,567.59,OTP,0.1279,0,0,64.78,0.2105
4,TXN_21630,USER_1000,39.41,Bank Transfer,2023-12-03 06:43:00,2023-10-29 03:47:00,842.933333,70995.83,Laptop,Mumbai,...,1,Visa,73,4851.37,PIN,0.6365,1,0,383.60,0.0000


In [27]:
def check_small_large(row):

    risk_increment = 0.0 
    
    # Filter out invalid time differences or transactions that occurred more than 1 hour apart
    if pd.isna(row['time_diff_hours']) or row['time_diff_hours'] >= 1:
        return risk_increment
    # Handle ZeroDivisionError cases
    if pd.isna(row['prev_amount']) or row['prev_amount'] <= 0:
        return risk_increment
        
    # Calculate the scale of price gap
    amount_ratio = row['Transaction_Amount'] / row['prev_amount']
    
    # Ignore transactions unless the current amount is at least 10 times larger than the previous one
    if amount_ratio < 10.0:
        return risk_increment
        
    # Map the amount explosion ratio to a 0-1 risk score using a sigmoid function
    ratio_limit = 50.0  # Midpoint threshold where risk hits 0.5
    k_amount = 0.05     # Slope sensitivity
    risk_score = 1 / (1 + np.exp(-k_amount * (amount_ratio - ratio_limit)))
    
    # Apply Time Decay: The shorter the time gap, the higher the urgency and risk factor
    time_factor = np.exp(-3.0 * row['time_diff_hours'])
    risk_score_final = risk_score * time_factor
    
    # Multiply risk for high-fraud categories (Electronics, Travel)
    if row['Merchant_Category'] in ['Electronics', 'Travel']:
        risk_score_final *= 1.3

    risk_increment = round(min(risk_score_final, 1.0), 4) 
    
    return risk_increment

# Update risk_score_new based on the 3th fraud detection feature
df_small_large['risk_score_new'] += df_small_large.apply(check_small_large, axis=1)

In [28]:
df_small_large[df_small_large['risk_score_new'] > 0]

,Transaction_ID,User_ID,Transaction_Amount,Transaction_Type,Timestamp,prev_timestamp,time_diff_hours,Account_Balance,Device_Type,Location,...,Failed_Transaction_Count_7d,Card_Type,Card_Age,Transaction_Distance,Authentication_Method,Risk_Score,Is_Weekend,Fraud_Label,prev_amount,risk_score_new
1,TXN_28650,USER_1000,191.87,Online,2023-04-26 23:17:00,2023-02-21 10:25:00,1548.866667,4201.83,Laptop,Tokyo,...,2,Discover,14,3616.72,OTP,0.1897,0,0,69.02,0.2155
2,TXN_40117,USER_1000,64.78,Online,2023-09-01 20:09:00,2023-04-26 23:17:00,3068.866667,25291.69,Tablet,London,...,2,Discover,133,2042.18,Password,0.9632,1,1,191.87,0.4365
3,TXN_13680,USER_1000,383.60,Bank Transfer,2023-10-29 03:47:00,2023-09-01 20:09:00,1375.633333,53558.88,Laptop,Mumbai,...,1,Amex,3,567.59,OTP,0.1279,0,0,64.78,0.2105
6,TXN_31869,USER_1001,150.26,Online,2023-02-20 18:12:00,2023-01-25 12:16:00,629.933333,97356.04,Laptop,Tokyo,...,0,Discover,54,2626.52,OTP,0.2736,0,0,159.85,0.0003
7,TXN_16009,USER_1001,9.65,Bank Transfer,2023-03-18 06:17:00,2023-02-20 18:12:00,612.083333,58740.69,Tablet,London,...,3,Visa,28,501.77,PIN,0.6659,0,0,150.26,0.6679
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49799,TXN_36549,USER_9998,336.28,POS,2023-08-22 22:43:00,2023-08-22 17:22:00,5.350000,55490.35,Tablet,London,...,0,Visa,41,3483.61,PIN,0.1581,1,0,21.80,1.0013
49801,TXN_20102,USER_9998,14.89,Online,2023-09-30 17:47:00,2023-09-04 17:40:00,624.116667,46190.69,Tablet,New York,...,1,Mastercard,212,4546.61,OTP,0.7063,0,0,90.20,0.4778
49802,TXN_11163,USER_9998,49.53,Bank Transfer,2023-11-04 04:01:00,2023-09-30 17:47:00,826.233333,81626.81,Tablet,New York,...,1,Amex,34,4727.77,OTP,0.2499,0,0,14.89,0.4775
49803,TXN_36472,USER_9998,228.91,Online,2023-12-13 09:28:00,2023-11-04 04:01:00,941.450000,34061.29,Mobile,Mumbai,...,3,Amex,222,3623.02,OTP,0.4687,0,0,49.53,0.0003


In [29]:
# 4th Fraud Feature: Account takeover
# Copy default dataframe
df_account_take = df_default.copy()

In [30]:
user_profile = df_account_take.groupby('User_ID').agg({
    'Device_Type': lambda x: x.mode()[0] if not x.mode().empty else None,
    'Authentication_Method': lambda x: x.mode()[0] if not x.mode().empty else None,
    'Transaction_Distance': 'mean'
}).reset_index()

user_profile.columns = ['User_ID', 'usual_device', 'usual_auth', 'avg_distance']

# Bring 'risk_score_new' from df_small_large and merge
df_account_take = pd.merge(df_account_take, user_profile, on='User_ID', how='left')
df_account_take = pd.merge(
    df_account_take.drop(columns=['risk_score_new']),
    df_small_large[['Transaction_ID', 'risk_score_new']],
    on='Transaction_ID',
    how='left'
)

df_account_take = reorder_columns(df_account_take)

In [31]:
def check_account_take(row):
    
    risk_increment = 0.0
    
    # Check if the user failed transaction more than 3 times
    if row['Failed_Transaction_Count_7d'] > 3:
        
        # Indicator A: Authentication method change
        if row['Authentication_Method'] != row['usual_auth']:
            risk_increment += 0.20
            
        # Indicator B: Device type change
        if row['Device_Type'] != row['usual_device']:
            risk_increment += 0.20
            
        # Indicator C: Network & Location Anomaly
        ip_anomaly = (row['IP_Address_Flag'] == 1)
        distance_anomaly = False

        if not pd.isna(row['Transaction_Distance']) and not pd.isna(row['avg_distance']):
            distance_anomaly = (row['Transaction_Distance'] > row['avg_distance'] * 3)
        if ip_anomaly and distance_anomaly:
            risk_increment += 0.50  
        elif ip_anomaly or distance_anomaly:
            risk_increment += 0.20 
        
        return risk_increment

# Update risk_score_new based on the 4th fraud detection feature 
df_account_take['risk_score_new'] += df_account_take.apply(check_account_take, axis=1)

In [32]:
df_account_take[df_account_take['risk_score_new'] > 0]

,Transaction_ID,User_ID,Transaction_Amount,Transaction_Type,Timestamp,Account_Balance,Device_Type,usual_device,Location,Merchant_Category,...,Card_Type,Card_Age,Transaction_Distance,avg_distance,Authentication_Method,usual_auth,Risk_Score,Is_Weekend,Fraud_Label,risk_score_new
0,TXN_36059,USER_1000,69.02,Online,2023-02-21 10:25:00,36077.57,Tablet,Laptop,London,Groceries,...,Amex,170,2983.85,2812.342000,Password,OTP,0.5529,0,1,0.4000
12,TXN_23318,USER_1002,4.91,POS,2023-02-23 04:47:00,18258.28,Mobile,Mobile,Mumbai,Clothing,...,Amex,2,1914.92,2767.548000,Biometric,Biometric,0.6928,1,1,0.2656
13,TXN_35075,USER_1002,90.49,POS,2023-04-13 05:28:00,44820.92,Tablet,Mobile,Sydney,Groceries,...,Mastercard,236,1009.32,2767.548000,PIN,Biometric,0.0824,0,1,0.5553
17,TXN_39240,USER_1003,52.30,POS,2023-01-25 05:43:00,16113.80,Tablet,Laptop,London,Restaurants,...,Visa,143,24.85,1600.077500,OTP,OTP,0.8465,0,1,0.2000
18,TXN_40680,USER_1003,11.40,Bank Transfer,2023-04-01 15:23:00,23343.55,Laptop,Laptop,Tokyo,Clothing,...,Visa,177,2872.56,1600.077500,PIN,OTP,0.0746,0,1,0.4830
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49780,TXN_3813,USER_9996,11.00,Online,2023-09-29 08:45:00,6370.97,Laptop,Laptop,London,Groceries,...,Discover,48,666.83,2454.315000,Biometric,Biometric,0.0272,0,1,0.3623
49782,TXN_32244,USER_9996,128.28,Bank Transfer,2023-11-06 10:22:00,64674.15,Laptop,Laptop,Tokyo,Clothing,...,Discover,94,2853.04,2454.315000,Password,Biometric,0.9595,0,1,0.2746
49786,TXN_5312,USER_9997,74.25,Bank Transfer,2023-08-29 06:10:00,72696.81,Mobile,Tablet,Sydney,Clothing,...,Mastercard,107,722.35,2728.845000,Biometric,Biometric,0.9378,0,1,0.5711
49789,TXN_30347,USER_9998,5.74,POS,2023-01-21 03:53:00,17164.18,Laptop,Laptop,Sydney,Clothing,...,Visa,79,111.45,2561.803125,OTP,PIN,0.0001,1,1,0.5275


In [33]:
# Set threshold to classify fraud (0 or 1)
threshold = 0.6
df_account_take['fraud_label_new'] = (df_account_take['risk_score_new'] >= threshold).astype(int)

actual_fraud = df_account_take[df_account_take['Fraud_Label'] == 1]
matched_fraud = actual_fraud[actual_fraud['fraud_label_new'] == 1]

# Result analysis
# 1. Overall accuracy : Percentage of correctly predicted rows
accuracy = (df_account_take['Fraud_Label'] == df_account_take['fraud_label_new']).mean() * 100

# 2. Fraud detection rate : Percentage of actual fraud cases caught by the logic
if len(actual_fraud) > 0:
    recall = (len(matched_fraud) / len(actual_fraud)) * 100
else:
    recall = 0

# Print out the result
print(f"--- Result (Threshold: {threshold}) ---")
print(f"1. Overall accuracy: {accuracy:.2f}%")
print(f"2. Fraud dectection rate : {recall:.2f}%")
print(f"\n")
print(f"- Actual fraud count: {len(actual_fraud)} cases")
print(f"- Detected fraud count: {len(matched_fraud)} cases")

--- Result (Threshold: 0.6) ---
1. Overall accuracy: 71.85%
2. Fraud dectection rate : 12.40%


- Actual fraud count: 16004 cases
- Detected fraud count: 1985 cases


In [34]:
df_missed_fraud = df_account_take[
    (df_account_take['Fraud_Label'] == 1) & (df_account_take['fraud_label_new'] == 0)
]

print(f" 1. Missed fraud cases : {len(df_missed_fraud)} cases")
display(df_missed_fraud.head())

 1. Missed fraud cases : 14019 cases


,Transaction_ID,User_ID,Transaction_Amount,Transaction_Type,Timestamp,Account_Balance,Device_Type,usual_device,Location,Merchant_Category,...,Card_Age,Transaction_Distance,avg_distance,Authentication_Method,usual_auth,Risk_Score,Is_Weekend,Fraud_Label,risk_score_new,fraud_label_new
0,TXN_36059,USER_1000,69.02,Online,2023-02-21 10:25:00,36077.57,Tablet,Laptop,London,Groceries,...,170,2983.85,2812.342000,Password,OTP,0.5529,0,1,0.4000,0
2,TXN_40117,USER_1000,64.78,Online,2023-09-01 20:09:00,25291.69,Tablet,Laptop,London,Clothing,...,133,2042.18,2812.342000,Password,OTP,0.9632,1,1,NaN,0
5,TXN_1454,USER_1001,159.85,ATM Withdrawal,2023-01-25 12:16:00,73063.15,Tablet,Tablet,Sydney,Clothing,...,224,891.88,1554.188571,PIN,PIN,0.8944,1,1,NaN,0
8,TXN_19073,USER_1001,85.67,ATM Withdrawal,2023-09-16 07:14:00,15337.07,Tablet,Tablet,Tokyo,Electronics,...,56,2952.58,1554.188571,Biometric,PIN,0.8601,1,1,NaN,0
12,TXN_23318,USER_1002,4.91,POS,2023-02-23 04:47:00,18258.28,Mobile,Mobile,Mumbai,Clothing,...,2,1914.92,2767.548000,Biometric,Biometric,0.6928,1,1,0.2656,0


In [35]:
df_false_alerts = df_account_take[
    (df_account_take['Fraud_Label'] == 0) & (df_account_take['fraud_label_new'] == 1)
]
print(f"\n 2. False alerts : {len(df_false_alerts)} cases")
display(df_false_alerts.head())


 2. False alerts : 0 cases


,Transaction_ID,User_ID,Transaction_Amount,Transaction_Type,Timestamp,Account_Balance,Device_Type,usual_device,Location,Merchant_Category,...,Card_Age,Transaction_Distance,avg_distance,Authentication_Method,usual_auth,Risk_Score,Is_Weekend,Fraud_Label,risk_score_new,fraud_label_new
